# Week 8 Capstone: Customer Churn Business Analysis
**Objective:** identify at-risk customer segments and recommend measurable retention interventions.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score
df = pd.read_csv('data/processed/customer_churn_cleaned.csv')
display(df.head())
print('Rows:',len(df),'Churn rate:',df.Churn.mean())

In [ ]:
# Segment analysis
contract = df.groupby('Contract').Churn.agg(['mean','count']).sort_values('mean',ascending=False)
tenure_bins=pd.cut(df.Tenure,[-1,6,12,24,48,1000],labels=['0-6','7-12','13-24','25-48','49+'])
tenure=df.groupby(tenure_bins,observed=False).Churn.agg(['mean','count'])
display(contract,tenure)

In [ ]:
# Statistical tests
print('Contract vs churn:',stats.chi2_contingency(pd.crosstab(df.Contract,df.Churn)))
print('Tenure by churn:',stats.ttest_ind(df.loc[df.Churn==0,'Tenure'],df.loc[df.Churn==1,'Tenure'],equal_var=False))
print('Payment vs churn:',stats.chi2_contingency(pd.crosstab(df.PaymentMethod,df.Churn)))

In [ ]:
# Exploratory logistic regression
num=['Tenure','MonthlyCharges','TotalCharges','SeniorCitizen']
cat=['Contract','PaymentMethod','PaperlessBilling']
pre=ColumnTransformer([('num',Pipeline([('imp',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),
                       ('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
pipe=Pipeline([('prep',pre),('model',LogisticRegression(max_iter=1500,class_weight='balanced',random_state=42))])
X=df[num+cat]; y=df.Churn
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
pipe.fit(Xtr,ytr); p=pipe.predict(Xte)
print(classification_report(yte,p))
print('ROC-AUC:',roc_auc_score(yte,pipe.predict_proba(Xte)[:,1]))

## Recommendations
Focus onboarding during months 0–12, pilot optional longer-contract offers for month-to-month customers, investigate payment friction, and measure 90/180-day retention in a controlled test. Treat the model as exploratory until validated on a newer sample.